In [1]:
# 0330版本fsfp处理（加入退市截断）
from pathlib import Path
import re
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd

def read_first_sheet_xlsx(path):
    ns = {
        'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main',
        'rel': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships',
        'pkg_rel': 'http://schemas.openxmlformats.org/package/2006/relationships',
    }

    def column_index(cell_ref):
        letters = re.match(r'^[A-Z]+', cell_ref).group(0)
        idx = 0
        for char in letters:
            idx = idx * 26 + ord(char) - ord('A') + 1
        return idx - 1

    with zipfile.ZipFile(path) as archive:
        shared_strings = []
        if 'xl/sharedStrings.xml' in archive.namelist():
            root = ET.fromstring(archive.read('xl/sharedStrings.xml'))
            for item in root.findall('main:si', ns):
                texts = [node.text or '' for node in item.findall('.//main:t', ns)]
                shared_strings.append(''.join(texts))

        workbook = ET.fromstring(archive.read('xl/workbook.xml'))
        first_sheet = workbook.find('main:sheets/main:sheet', ns)
        first_sheet_rel_id = first_sheet.attrib['{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id']

        rels = ET.fromstring(archive.read('xl/_rels/workbook.xml.rels'))
        target = None
        for rel in rels.findall('pkg_rel:Relationship', ns):
            if rel.attrib['Id'] == first_sheet_rel_id:
                target = rel.attrib['Target']
                break
        if target is None:
            raise ValueError('无法在 xlsx 中定位第一张工作表')

        sheet_path = 'xl/' + target.lstrip('/')
        sheet = ET.fromstring(archive.read(sheet_path))
        rows = []
        max_col = 0
        for row in sheet.findall('main:sheetData/main:row', ns):
            values = {}
            for cell in row.findall('main:c', ns):
                col_idx = column_index(cell.attrib['r'])
                max_col = max(max_col, col_idx)
                value_node = cell.find('main:v', ns)
                inline_node = cell.find('main:is/main:t', ns)
                if value_node is None and inline_node is None:
                    value = None
                elif cell.attrib.get('t') == 's':
                    value = shared_strings[int(value_node.text)]
                elif cell.attrib.get('t') == 'inlineStr':
                    value = inline_node.text if inline_node is not None else None
                else:
                    value = value_node.text if value_node is not None else None
                    if value is not None:
                        value = pd.to_numeric(value, errors='ignore')
                values[col_idx] = value
            rows.append([values.get(i) for i in range(max_col + 1)])

    header = rows[0]
    data = [(row + [None] * len(header))[:len(header)] for row in rows[1:]]
    return pd.DataFrame(data, columns=header)

def parse_mixed_date(series):
    parsed = pd.to_datetime(series, errors='coerce', format='mixed')
    numeric = pd.to_numeric(series, errors='coerce')
    looks_like_excel_serial = numeric.between(20000, 60000).mean() > 0.5
    if looks_like_excel_serial:
        excel_dates = pd.to_datetime(numeric, unit='D', origin='1899-12-30', errors='coerce')
        parsed = excel_dates.fillna(parsed)
    return parsed

def find_col(df, candidates, label):
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"未找到{label}列，候选列: {candidates}，当前列: {list(df.columns)}")

# =========================
# 1. 读取数据
# =========================
base_dir = Path.cwd()
if not (base_dir / 'data').exists() and (base_dir / '代码汇总' / 'data').exists():
    base_dir = base_dir / '代码汇总'

data_dir = base_dir / 'data'
output_dir = base_dir / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

fsfp_path = data_dir / 'FSFP_词频' / 'fsfp_weekly_filled_0509.csv'
afco_path = data_dir / 'AF_Co.xlsx'
delist_candidates = [data_dir / '退市.csv', data_dir / '退市(1).csv']
delist_path = next((path for path in delist_candidates if path.exists()), None)
if delist_path is None:
    raise FileNotFoundError(f"未找到退市文件，已检查: {[str(path) for path in delist_candidates]}")
output_path = output_dir / 'fsfp_label.csv'

fsfp = pd.read_csv(fsfp_path, dtype={'code': str, 'week': str})
required_fsfp_cols = {'code', 'week', 'fsfp_original'}
missing_fsfp_cols = required_fsfp_cols - set(fsfp.columns)
if missing_fsfp_cols:
    raise ValueError(f"FSFP 输入缺少必要字段: {sorted(missing_fsfp_cols)}")
fsfp = fsfp[['code', 'week', 'fsfp_original']].copy()

try:
    afco = pd.read_excel(afco_path)
except ImportError:
    afco = read_first_sheet_xlsx(afco_path)
delist = pd.read_csv(delist_path, encoding='utf-8-sig')

# =========================
# 2. 统一股票代码格式
# =========================
afco_code_col = find_col(afco, ['证券代码', 'Stkcd', '股票代码_StkCd', 'code'], 'AF_Co 股票代码')
afco_list_date_col = find_col(afco, ['首次上市日期', 'Listdt'], 'AF_Co 首次上市日期')
delist_code_col = find_col(delist, ['股票代码_StkCd', 'Stkcd', '证券代码', 'code'], '退市股票代码')
delist_date_col = find_col(delist, ['退市日期_DelstDt', 'DelstDt', '退市日期'], '退市日期')

fsfp['code'] = fsfp['code'].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(6)
fsfp['week'] = pd.to_numeric(fsfp['week'], errors='coerce')
fsfp['fsfp_original'] = pd.to_numeric(fsfp['fsfp_original'], errors='coerce')
afco['code'] = afco[afco_code_col].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(6)
delist['code'] = delist[delist_code_col].astype(str).str.extract(r'(\d+)', expand=False).str.zfill(6)

# =========================
# 3. 处理日期
# =========================
afco['首次上市日期'] = parse_mixed_date(afco[afco_list_date_col])
delist['退市日期_DelstDt'] = parse_mixed_date(delist[delist_date_col])

afco = afco[afco['code'].str.match(r'^\d{6}$', na=False)].copy()
delist = delist[delist['code'].str.match(r'^\d{6}$', na=False)].copy()

# =========================
# 4. 日期 -> week编码
#    格式: 201501 表示 2015年第1周
# =========================
def date_to_week_code(dt):
    if pd.isna(dt):
        return pd.NA
    iso = dt.isocalendar()
    return int(f"{iso.year}{iso.week:02d}")

afco['start_week'] = afco['首次上市日期'].apply(date_to_week_code)
delist['delist_week'] = delist['退市日期_DelstDt'].apply(date_to_week_code)

# =========================
# 5. 取 AF_Co 中 1-5400 的股票
#    若有重复证券代码，只保留第一次出现
# =========================
afco_selected = pd.concat([
    afco.iloc[0:5400]    # 1-5400
], axis=0, ignore_index=True)

afco_selected = (
    afco_selected[['code', 'start_week']]
    .drop_duplicates(subset=['code'], keep='first')
    .copy()
)

# 起始周下限截断为 201501
afco_selected['start_week'] = afco_selected['start_week'].clip(lower=201501)

# =========================
# 6. 整理退市信息
#    若同一股票有多条退市记录，保留最早退市周
# =========================
delist_info = (
    delist[['code', 'delist_week']]
    .dropna(subset=['code', 'delist_week'])
    .sort_values(['code', 'delist_week'])
    .drop_duplicates(subset=['code'], keep='first')
    .copy()
)

# 合并到选中股票表
afco_selected = afco_selected.merge(delist_info, on='code', how='left')

# 统一最大结束周
global_end_week = 202529

# 对每只股票确定最终结束周
afco_selected['end_week'] = pd.to_numeric(afco_selected['delist_week'], errors='coerce')
afco_selected['end_week'] = afco_selected['end_week'].fillna(global_end_week)
afco_selected['end_week'] = afco_selected['end_week'].clip(upper=global_end_week)

# =========================
# 7. 处理 fsfp
#    若同一股票同一周有重复，保留最后一条
# =========================
fsfp = (
    fsfp[['code', 'week', 'fsfp_original']]
    .dropna(subset=['code', 'week'])
    .sort_values(['code', 'week'])
    .drop_duplicates(subset=['code', 'week'], keep='last')
)

fsfp['week'] = fsfp['week'].astype(int)
fsfp['fsfp_original'] = fsfp['fsfp_original'].fillna(0)
afco_selected['start_week'] = afco_selected['start_week'].astype(int)
afco_selected['end_week'] = afco_selected['end_week'].astype(int)

# =========================
# 8. 构造完整周序列：201501 ~ 202529
# =========================
all_mondays = pd.date_range(start='1999-12-29', end='2025-07-14', freq='W-MON')
all_weeks = [date_to_week_code(d) for d in all_mondays]
all_weeks = [int(w) for w in all_weeks if 201501 <= w <= global_end_week]

# =========================
# 9. 为选中股票逐个生成完整面板
#    截断到该公司 end_week
# =========================
panel_list = []

for _, row in afco_selected.iterrows():
    code = row['code']
    start_week = row['start_week']
    end_week = row['end_week']

    if end_week < start_week:
        continue

    valid_weeks = [w for w in all_weeks if start_week <= w <= end_week]

    tmp = pd.DataFrame({
        'code': code,
        'week': valid_weeks
    })
    panel_list.append(tmp)

panel = pd.concat(panel_list, ignore_index=True)

# =========================
# 10. 合并并补0
# =========================
result = panel.merge(fsfp, on=['code', 'week'], how='left')
result['fsfp_original'] = result['fsfp_original'].fillna(0)

nonzero_median = result.loc[result['fsfp_original'] != 0, 'fsfp_original'].median()

result['FSFP'] = 0
result.loc[
    (result['fsfp_original'] > 0) & (result['fsfp_original'] <= nonzero_median),
    'FSFP'
] = 1
result.loc[
    result['fsfp_original'] > nonzero_median,
    'FSFP'
] = 2

# =========================
# 11. 排序导出 + 统计输出
# =========================
result = result.sort_values(['code', 'week']).reset_index(drop=True)
result.to_csv(output_path, index=False, encoding='utf-8-sig')

fsfp_counts = result['FSFP'].value_counts().sort_index()
count_0 = fsfp_counts.get(0, 0)
count_1 = fsfp_counts.get(1, 0)
count_2 = fsfp_counts.get(2, 0)
nonzero_ratio = 1 - count_0 / len(result)

print('处理完成，输出文件：', output_path)
print('非零值中位数 =', nonzero_median)
print('FSFP=0 的个数:', count_0)
print('FSFP=1 的个数:', count_1)
print('FSFP=2 的个数:', count_2)
print('FSFP非0比例 = {:.6f}'.format(nonzero_ratio))
print('FSFP非0比例(%) = {:.4f}%'.format(nonzero_ratio * 100))
print(result.head())
print(result.tail())
print('股票数:', result['code'].nunique())
print('总样本数:', len(result))

处理完成，输出文件： C:\Users\chenyu\Desktop\财务造假\代码汇总\output\fsfp_label.csv
非零值中位数 = 1.647364845126133
FSFP=0 的个数: 1044440
FSFP=1 的个数: 579310
FSFP=2 的个数: 579308
FSFP非0比例 = 0.525914
FSFP非0比例(%) = 52.5914%
     code    week  fsfp_original  FSFP
0  000001  201501            0.0     0
1  000001  201502            0.0     0
2  000001  201503            0.0     0
3  000001  201504            0.0     0
4  000001  201505            0.0     0
           code    week  fsfp_original  FSFP
2203053  688583  202525            0.0     0
2203054  688583  202526            0.0     0
2203055  688583  202527            0.0     0
2203056  688583  202528            0.0     0
2203057  688583  202529            0.0     0
股票数: 5312
总样本数: 2203058
